<a href="https://colab.research.google.com/github/esthy13/Water-Level-Distribution/blob/main/water_level_prediction_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Water level prediction using ERA5

This notebook implements a sequence model that predicts water level at each node using only **past** inputs:
- Node latitude/longitude (static)
- Ephemerides (time-varying, global)
- ERA5 weather (time-varying, grid-based u10/v10/sp)

**Important:** Water level is never used as input (not even past values).

In [3]:
import numpy as np
from datetime import datetime
from tqdm import tqdm
import os

os.environ["KERAS_BACKEND"] = "torch"  # optional; use torch backend if available

import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

In [ ]:
import gdown
!gdown 1NzYayIEoY_JzW9E-38ejnHahNlALlFBn
!gdown 1viecawNSkSEGUvEk7nu7rhKcRd5Q7wgm
!gdown 1aBCiXHEWN7ZeIlyj77qNCZ7DHwCUy9LG
!gdown 19m4hBBrup_Ast_ozGFDmVK-i7N2DUQC5
!gdown 1wtRof_MQJlE2GNplfbflAPR3cxpmiCeZ
!gdown 13q1PVFDcOTujx_r4fMPKxkJeMw4fAt5j
!gdown 1wVmxNvCg6Pe9Ol3LGM1NykXVwJiZUdE4
!gdown 1osjH0XjabPwZNuN8hKK1TcIvQ63Cfyda
!gdown 1epCvAsWYrCISdcbc2dXZArow5ouhsWjl
!gdown 1sWoTlJih-mqDdP9TBzhyXTqHHS5Jr9fe

Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1NzYayIEoY_JzW9E-38ejnHahNlALlFBn

but Gdown can't. Please check connections and permissions.
Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1viecawNSkSEGUvEk7nu7rhKcRd5Q7wgm

but Gdown can't. Please check connections and permissions.


## Data loading

In [ ]:
# load nodes coordinates
lat_vec = np.load("./lat.npy") # (5000, )
lon_vec = np.load("./lon.npy") # (5000, )

#load train data
train_wl = np.load("./wl_2010-2020.npy") # (87648, 5000)
train_ephem = np.load("./dist_alt_az_moon-sun_coord13-45_2010-2019_norm.npy") # (6, 87648)
train_era5 = np.load("./ERA5_adriatic_u10v10sp_2010-2019.npy") # (3, 87648, 5, 9)
train_tvec = np.load("./tvec_2010-2019.npy") # (6, 87648))

#load test data
test_wl = np.load("./wl_2020.npy") # (8784, 5000)
test_ephem = np.load("./dist_alt_az_moon-sun_coord13-45_2020_norm.npy") # (6, 8784)
test_era5 = np.load("./ERA5_adriatic_u10v10sp_2020.npy")#(3, 8784, 5, 9)
test_tvec = np.load("./tvec_2020.npy") # (6,8784)

T_train = train_wl.shape[0]
N_nodes = train_wl.shape[1]

print("Train hours:", T_train, "Nodes:", N_nodes)
print("Test hours:", test_wl.shape[0])

## Utilities

In [ ]:
def get_era5_coord(lat, lon):
    """
    Map a node coordinate to the ERA5 5x9 grid.
    """
    era5_row, era5_col = 5, 9
    lat_min, lat_max = 44.94972, 45.8
    lon_min, lon_max = 12.12863, 13.81283

    delta_lat = lat_max - lat_min
    delta_lon = lon_max - lon_min

    lon_coord = np.ceil((lon - lon_min) / delta_lon * (era5_col - 1))
    lat_coord = 4 - np.ceil((lat - lat_min) / delta_lat * (era5_row - 1))

    return int(lat_coord), int(lon_coord)

def RMSE(y_true, y_pred):
    return np.sqrt(np.mean((y_pred - y_true) ** 2))

def normalize_global(x, x_min=None, x_max=None):
    if x_min is None:
        x_min = x.min()
    if x_max is None:
        x_max = x.max()
    return (x - x_min) / (x_max - x_min + 1e-12), x_min, x_max

## Normalize ERA5 using train stats (important)

We normalize ERA5 with training min/max, then apply the same to test.

In [ ]:
train_era5_norm, era5_min, era5_max = normalize_global(train_era5)
test_era5_norm, _, _ = normalize_global(test_era5, era5_min, era5_max)

## Build input sequences

For each node and each time **t**, we use a past window of length `seq_len` of inputs:
- 6 ephemerides (global)
- 3 ERA5 values at node grid cell (u10, v10, sp)
- 2 static node coords (lat, lon) repeated across time

This yields a per-time feature size of **6 + 3 + 2 = 11**.

**Important:** no water levels are used as inputs.

In [ ]:
seq_len = 24  # 24 hours of past inputs

# Train/val split: last 8760 hours of 2019 for validation
val_hours = 8760
train_end = T_train - val_hours

# Precompute node ERA5 coords
node_era5_coords = np.array([get_era5_coord(lat_vec[i], lon_vec[i]) for i in range(N_nodes)])

def build_sequences(ephem, era5, t_start, t_end, seq_len):
    """
    Build sequences for all nodes between t_start and t_end (exclusive).
    Returns X (samples, seq_len, features) and Y (samples, ) for one node at a time.
    """
    # ephem: (6, T)
    # era5: (3, T, 5, 9)
    T = ephem.shape[1]
    assert t_start >= seq_len
    assert t_end <= T
    n_samples = (t_end - t_start)
    return n_samples

def make_node_dataset(node_idx, ephem, era5, wl, t_start, t_end, seq_len):
    """
    Build (X, y) for a single node.
    """
    lat = lat_vec[node_idx]
    lon = lon_vec[node_idx]
    r, c = node_era5_coords[node_idx]

    n_samples = t_end - t_start
    X = np.zeros((n_samples, seq_len, 11), dtype=np.float32)
    y = np.zeros((n_samples,), dtype=np.float32)

    for i, t in enumerate(range(t_start, t_end)):
        # Past window: [t-seq_len, t)
        ephem_window = ephem[:, t-seq_len:t].T  # (seq_len, 6)
        era5_window = era5[:, t-seq_len:t, r, c].T  # (seq_len, 3)
        latlon = np.repeat(np.array([[lat, lon]], dtype=np.float32), seq_len, axis=0)  # (seq_len, 2)

        X[i] = np.concatenate([ephem_window, era5_window, latlon], axis=1)
        y[i] = wl[t, node_idx]

    return X, y

## Model

We use an LSTM-based regressor to process temporal sequences.

In [ ]:
def build_model(seq_len, n_features):
    inp = layers.Input(shape=(seq_len, n_features))
    x = layers.LSTM(64, return_sequences=True)(inp)
    x = layers.LSTM(32)(x)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(1)(x)

    model = models.Model(inp, out)
    model.compile(optimizer='adam', loss='mse')
    return model

model = build_model(seq_len, 11)
model.summary()

## Training strategy

We train on a subset of nodes to keep memory and time manageable. You can increase `train_nodes` for better performance.

Training is done on 2010–2018 (train) and 2019 (validation).

In [ ]:
train_nodes = 128  # adjust up for better accuracy
train_node_ids = np.random.choice(N_nodes, size=train_nodes, replace=False)

X_train_list, y_train_list = [], []
X_val_list, y_val_list = [], []

for node_idx in tqdm(train_node_ids, desc="Building datasets"):
    # Train: [seq_len, train_end)
    Xn, yn = make_node_dataset(
        node_idx,
        train_ephem,
        train_era5_norm,
        train_wl,
        t_start=seq_len,
        t_end=train_end,
        seq_len=seq_len
    )
    # Val: [train_end, T_train)
    Xv, yv = make_node_dataset(
        node_idx,
        train_ephem,
        train_era5_norm,
        train_wl,
        t_start=train_end,
        t_end=T_train,
        seq_len=seq_len
    )
    X_train_list.append(Xn)
    y_train_list.append(yn)
    X_val_list.append(Xv)
    y_val_list.append(yv)

X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)
X_val = np.concatenate(X_val_list, axis=0)
y_val = np.concatenate(y_val_list, axis=0)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=256,
    verbose=1
)

## Evaluate on 2020 test set

We evaluate on a subset of nodes (same subset) to compute RMSE.

In [ ]:
def evaluate_on_test(nodes):
    rmses = []
    for node_idx in tqdm(nodes, desc="Testing"):
        Xt, yt = make_node_dataset(
            node_idx,
            test_ephem,
            test_era5_norm,
            test_wl,
            t_start=seq_len,
            t_end=test_wl.shape[0],
            seq_len=seq_len
        )
        pred = model.predict(Xt, batch_size=256, verbose=0).squeeze()
        rmse = RMSE(yt, pred)
        rmses.append(rmse)
    return np.mean(rmses), np.std(rmses)

mean_rmse, std_rmse = evaluate_on_test(train_node_ids)
print(f"Test RMSE (mean over {len(train_node_ids)} nodes): {mean_rmse:.4f} ± {std_rmse:.4f}")

## Persistence baseline

This uses *previous water level* and is **not allowed as a model input**, but is used here only to compute the reference score.

In [ ]:
print(f"Persistence baseline (test): {RMSE(test_wl[:-1], test_wl[1:]):.4f}")